In [0]:
from pyspark.sql.functions import col, trim, upper, to_date


marketplace_bronze = spark.table("bronze_marketplace")


display(marketplace_bronze)

condition,listing_date,listing_id,listing_type,matched_sku,price,seller,title
Good,2026-07-01,M001,product,LAP1001,42000,Seller_A,EcoBook Pro 14 Laptop 16GB
Used,2026-07-03,M002,product,LAP1001,38000,Seller_B,EcoBook Pro14 i7 Used Laptop
Used,2026-07-05,M003,component,LAP1001,9000,Seller_C,Eco Book Pro 14 Screen Display
Used,2026-07-06,M004,component,LAP1001,4000,Seller_D,EcoBook Pro 14 Battery Original
For Parts,2026-07-07,M005,component,LAP1001,2200,Seller_E,EcoBook Pro 14 Motherboard
Good,2026-07-02,M006,product,LAP1002,36000,Seller_F,EcoBook Air 13 Laptop
Fair,2026-07-04,M007,product,LAP1002,32000,Seller_G,EcoBook Air13 13-inch Used
Used,2026-07-08,M008,component,LAP1002,8200,Seller_H,EcoBook Air 13 Display Panel
Good,2026-07-02,M009,product,LAP1003,26000,Seller_I,EcoBook Basic 15 Laptop
Used,2026-07-09,M010,product,LAP1003,23000,Seller_J,EcoBook Basic15 Used Notebook


In [0]:
marketplace_silver = marketplace_bronze \
    .withColumn("listing_id", upper(trim(col("listing_id")))) \
    .withColumn("matched_sku", upper(trim(col("matched_sku")))) \
    .withColumn("listing_type", trim(col("listing_type"))) \
    .withColumn("condition", trim(col("condition"))) \
    .withColumn("seller", trim(col("seller"))) \
    .withColumn("title", trim(col("title")))


In [0]:
from pyspark.sql.functions import regexp_replace


marketplace_silver = marketplace_silver.withColumn(
    "price",
    regexp_replace(trim(col("price")), ",", "").cast("double")
)

In [0]:
marketplace_silver = marketplace_silver.withColumn(
    "listing_date",
    to_date(col("listing_date"), "yyyy-MM-dd")
)

In [0]:
display(marketplace_silver)

condition,listing_date,listing_id,listing_type,matched_sku,price,seller,title
Good,2026-07-01,M001,product,LAP1001,42000.0,Seller_A,EcoBook Pro 14 Laptop 16GB
Used,2026-07-03,M002,product,LAP1001,38000.0,Seller_B,EcoBook Pro14 i7 Used Laptop
Used,2026-07-05,M003,component,LAP1001,9000.0,Seller_C,Eco Book Pro 14 Screen Display
Used,2026-07-06,M004,component,LAP1001,4000.0,Seller_D,EcoBook Pro 14 Battery Original
For Parts,2026-07-07,M005,component,LAP1001,2200.0,Seller_E,EcoBook Pro 14 Motherboard
Good,2026-07-02,M006,product,LAP1002,36000.0,Seller_F,EcoBook Air 13 Laptop
Fair,2026-07-04,M007,product,LAP1002,32000.0,Seller_G,EcoBook Air13 13-inch Used
Used,2026-07-08,M008,component,LAP1002,8200.0,Seller_H,EcoBook Air 13 Display Panel
Good,2026-07-02,M009,product,LAP1003,26000.0,Seller_I,EcoBook Basic 15 Laptop
Used,2026-07-09,M010,product,LAP1003,23000.0,Seller_J,EcoBook Basic15 Used Notebook


In [0]:
marketplace_silver.printSchema()

root
 |-- condition: string (nullable = true)
 |-- listing_date: date (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- listing_type: string (nullable = true)
 |-- matched_sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- seller: string (nullable = true)
 |-- title: string (nullable = true)



In [0]:
marketplace_silver.select(
    [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in marketplace_silver.columns
    ]
).show()

+---------+------------+----------+------------+-----------+-----+------+-----+
|condition|listing_date|listing_id|listing_type|matched_sku|price|seller|title|
+---------+------------+----------+------------+-----------+-----+------+-----+
|        0|           0|         0|           0|          0|    0|     0|    0|
+---------+------------+----------+------------+-----------+-----+------+-----+



In [0]:
marketplace_silver.groupBy("listing_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

marketplace_silver.filter(
    col("price") < 0
).show()


+----------+-----+
|listing_id|count|
+----------+-----+
+----------+-----+

+---------+------------+----------+------------+-----------+-----+------+-----+
|condition|listing_date|listing_id|listing_type|matched_sku|price|seller|title|
+---------+------------+----------+------------+-----------+-----+------+-----+
+---------+------------+----------+------------+-----------+-----+------+-----+



In [0]:
marketplace_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_marketplace")

In [0]:
display(spark.table("silver_marketplace"))

condition,listing_date,listing_id,listing_type,matched_sku,price,seller,title
Good,2026-07-01,M001,product,LAP1001,42000.0,Seller_A,EcoBook Pro 14 Laptop 16GB
Used,2026-07-03,M002,product,LAP1001,38000.0,Seller_B,EcoBook Pro14 i7 Used Laptop
Used,2026-07-05,M003,component,LAP1001,9000.0,Seller_C,Eco Book Pro 14 Screen Display
Used,2026-07-06,M004,component,LAP1001,4000.0,Seller_D,EcoBook Pro 14 Battery Original
For Parts,2026-07-07,M005,component,LAP1001,2200.0,Seller_E,EcoBook Pro 14 Motherboard
Good,2026-07-02,M006,product,LAP1002,36000.0,Seller_F,EcoBook Air 13 Laptop
Fair,2026-07-04,M007,product,LAP1002,32000.0,Seller_G,EcoBook Air13 13-inch Used
Used,2026-07-08,M008,component,LAP1002,8200.0,Seller_H,EcoBook Air 13 Display Panel
Good,2026-07-02,M009,product,LAP1003,26000.0,Seller_I,EcoBook Basic 15 Laptop
Used,2026-07-09,M010,product,LAP1003,23000.0,Seller_J,EcoBook Basic15 Used Notebook


In [0]:
print(
    "Bronze Marketplace records:",
    spark.table("bronze_marketplace").count()
)

print(
    "Silver Marketplace records:",
    spark.table("silver_marketplace").count()
)

Bronze Marketplace records: 26
Silver Marketplace records: 26
